In [0]:
%pip install pendulum

In [0]:
import json 
from pathlib import Path
import pendulum
import requests

In [0]:
current_catalog = spark.sql("SELECT current_catalog()").first()[0]

catalog = current_catalog
schema = "raw"
volume = "usgs_earthquakes"

print("Catálogo:" , catalog)
print("Schema:" , schema)
print("Volume:" , volume)



In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

print(f"Volume creado:  {catalog}-{schema}-{volume}")

In [0]:
#PRIMERA CARGA - FILTROS 2026
#dbutils.widgets.text("start_date", "2026-01-01","Fecha inicio")
#dbutils.widgets.text("end_date", "2026-08-29","Fecha fin")

dbutils.widgets.text(
    "output_dir",
    f"/Volumes/{catalog}/{schema}/{volume}",
    "Ruta raw",
)

dbutils.widgets.text(
    "api_endpoint",
    "https://earthquake.usgs.gov/fdsnws/event/1/query",
    "USGS API"
)

dbutils.widgets.text("min_magnitude","3.0","Magnitud mínima")
dbutils.widgets.text("timeout","60","timeout")

In [0]:
from datetime import datetime, timedelta

start_date = (datetime.now() - timedelta(days=5)).strftime("%Y-%m-%d")

end_date = datetime.now().strftime("%Y-%m-%d")

#CARGA INICIAL
#start_date = dbutils.widgets.get("start_date")
#end_date = dbutils.widgets.get("end_date")
output_dir = dbutils.widgets.get("output_dir")
api_endpoint = dbutils.widgets.get("api_endpoint")

min_magnitude = float(dbutils.widgets.get("min_magnitude"))
timeout = int(dbutils.widgets.get("timeout"))

print("Fecha inicio:", start_date)
print("Fecha fin:", end_date)
print("Magnitud mínima:", min_magnitude)
print("Ruta raw:", output_dir)
print("USGS API:", api_endpoint)
print("Timeout:", timeout)

In [0]:
region = {
    "minlatitude": -56.5,
    "maxlatitude": 0.5,
    "minlongitude": -85.0,
    "maxlongitude": -65.0
}
print(region)

In [0]:
params = {
    "format": "geojson",
    "starttime": start_date,
    "endtime": end_date,
    "minmagnitude": min_magnitude,
    "eventtype": "earthquake",
    "orderby": "time-asc",
    **region
}

response = requests.get(
    api_endpoint,
    params=params,
    timeout=timeout
)

response.raise_for_status()
data = response.json()

print("HTTP", response.status_code)
print("Eventos encontrados:", len(data["features"]))

In [0]:
#params = {
#    "format": "geojson",
#    "starttime": start_date,
#    "endtime": end_date,
#    "minmagnitude": min_magnitude,
#    "eventtype": "earthquake",
#    "orderby": "time-asc",
#    **zones["peru"] #cambiar por el nombre del pais de la lista zones
#}

#response = requests.get(
#    api_endpoint,
#    params=params,
#    timeout=timeout
#)

#response.raise_for_status()
#data = response.json()

#print("HTTP", response.status_code)
#print("Eventos encontrados:", len(data["features"]))

In [0]:
from pathlib import Path
from datetime import datetime
import requests

session = requests.Session()

params = {
    "format": "geojson",
    "starttime": start_date,
    "endtime": end_date,
    "minmagnitude": min_magnitude,
    "eventtype": "earthquake",
    "orderby": "time-asc",
    **region
}

response = session.get(
    api_endpoint,
    params=params,
    timeout=timeout
)

response.raise_for_status()
execution_date = datetime.now()
year = execution_date.strftime("%Y")
month = execution_date.strftime("%m")
day = execution_date.strftime("%d")

folder_path = (
    Path(output_dir)
    / year
    / month
    / day
)

folder_path.mkdir(
    parents=True,
    exist_ok=True
)

file_name = (f"earthquakes_{start_date}_{end_date}.json")

file_path = folder_path / file_name

file_path.write_bytes(response.content)

cantidad = len(response.json()["features"])

print(f"Raw eventos: {cantidad}")
